In [ ]:
!pip install ultralytics roboflow -q

from roboflow import Roboflow
rf = Roboflow(api_key="ULSlDKEuPmkczJNb4WjE")
project = rf.workspace("princes-workspace-2g1s7").project("red-scale-finetune")
version = project.version(1)
dataset = version.download("yolov8")

from IPython.display import clear_output
clear_output()
print("Dataset downloaded and Ultralytics installed!")

In [ ]:
import yaml

yaml_path = '/kaggle/working/Red-Scale-Finetune-1/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# 2. Update the paths to point exactly to your folders
# Since there is no 'valid' split, we point validation to 'train'
data['path'] = '/kaggle/working/Red-Scale-Finetune-1'
data['train'] = 'train/images'
data['val'] = 'train/images'  

# 3. Save the fixed yaml back
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("🔥 data.yaml paths fixed successfully! 🔥")
# Print it out to verify
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/input/datasets/princemridul/pt-ckpt/best_yolo.pt') 

results = model.train(
    data='/kaggle/working/Red-Scale-Finetune-1/data.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=4,      
    freeze=10,    
    name='red_scale_finetune'
)

print("Training Complete! Check the 'runs' folder for your new weights.")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load your brand new, fine-tuned weights
new_model = YOLO('/kaggle/working/runs/detect/red_scale_finetune-3/weights/best.pt')

# 2. Point to your training/validation image folder to see how it performs
IMAGE_DIR = '/kaggle/working/Red-Scale-Finetune-1/train/images'
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Let's test it out on the first 5 images visually
test_files = image_files[:5]
plt.figure(figsize=(15, 5 * len(test_files)))

for idx, file in enumerate(test_files):
    img_path = os.path.join(IMAGE_DIR, file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run inference with the new model
    res = new_model(img_rgb, conf=0.25, verbose=False)
    
    # Extract the custom crop from the newly predicted boxes
    boxes = res[0].boxes.xyxy.cpu().numpy()
    if len(boxes) > 0:
        x1, y1, x2, y2 = map(int, boxes[0])
        # Add a comfortable 5% padding so the digits don't hug the edge
        h_img, w_img = img_rgb.shape[:2]
        pad = int(h_img * 0.03)
        x1_pad = max(0, x1 - pad)
        y1_pad = max(0, y1 - pad)
        x2_pad = min(w_img, x2 + pad)
        y2_pad = min(h_img, y2 + pad)
        
        crop = img_rgb[y1_pad:y2_pad, x1_pad:x2_pad]
    else:
        crop = np.zeros((100, 100, 3), dtype=np.uint8) # Fallback if it misses
        
    # Plot the predicted bounding boxes overlay
    plt.subplot(len(test_files), 2, 2*idx + 1)
    plt.imshow(res[0].plot())
    plt.title(f"YOLO Detection: {file}")
    plt.axis("off")
    
    # Plot the isolated region going to the OCR model
    plt.subplot(len(test_files), 2, 2*idx + 2)
    plt.imshow(crop)
    plt.title("Extracted Screen ROI")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import torch
import re
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
from torchvision import transforms as T

# 1. Initialize PARSeq and transformations (if not already run in a previous cell)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).eval().to(DEVICE)

parseq_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5)
])

# 2. Helper filters for confidence scoring and decimal styling
def filter_by_confidence(raw_str, conf_tensor, threshold=0.40):
    return "".join([char for char, conf in zip(raw_str, conf_tensor) if conf.item() >= threshold])

def post_process_prediction(pred):
    clean = re.sub(r'[^0-9\.]', '', pred)
    if '.' not in clean and len(clean) >= 4:
        clean = clean[:-3] + '.' + clean[-3:]
    return clean

# 3. Load the fine-tuned model weights we just generated
new_model = YOLO('/kaggle/working/runs/detect/red_scale_finetune-3/weights/best.pt')

IMAGE_DIR = '/kaggle/working/Red-Scale-Finetune-1/train/images'
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Let's run and display the predictions for the first 5 images
test_files = image_files[:5]
plt.figure(figsize=(15, 5 * len(test_files)))


for idx, file in enumerate(test_files):
    img_path = os.path.join(IMAGE_DIR, file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run inference forcing Class 0 (the scale display) and dropping low-conf noise
    res = new_model(img_rgb, conf=0.60, classes=[0], verbose=False)
    
    boxes = res[0].boxes.xyxy.cpu().numpy()
    if len(boxes) > 0:
        x1, y1, x2, y2 = map(int, boxes[0])
        h_img, w_img = img_rgb.shape[:2]
        
        # Add slight padding for comfortable PARSeq visibility
        pad = int(h_img * 0.03)
        x1_pad = max(0, x1 - pad)
        y1_pad = max(0, y1 - pad)
        x2_pad = min(w_img, x2 + pad)
        y2_pad = min(h_img, y2 + pad)
        
        tight_crop = img_rgb[y1_pad:y2_pad, x1_pad:x2_pad]
        
        # Apply a light 2x2 dilation to clean up the LED segments for the OCR model
        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        processed_crop = cv2.dilate(tight_crop, kernel_dilate, iterations=1)
        
        # Run PARSeq sequence matching on the tight crop
        with torch.no_grad():
            tensor = parseq_transform(Image.fromarray(processed_crop)).unsqueeze(0).to(DEVICE)
            logits = parseq(tensor)
            decoded_strs, confidences = parseq.tokenizer.decode(logits.softmax(-1))
            
            raw_pred = decoded_strs[0].strip()
            char_confs = confidences[0]

        # Apply confidence parsing and formatting
        filtered_pred = filter_by_confidence(raw_pred, char_confs, threshold=0.40)
        final_prediction = post_process_prediction(filtered_pred)
    else:
        processed_crop = np.zeros((100, 100, 3), dtype=np.uint8)
        final_prediction = "SCALE NOT DETECTED"

    print(f"Image: {file} ---> Predicted Value: {final_prediction}")

    # Plot visual verification
    plt.subplot(len(test_files), 2, 2*idx + 1)
    plt.imshow(res[0].plot())
    plt.title(f"YOLO Box: {file}")
    plt.axis("off")
    
    plt.subplot(len(test_files), 2, 2*idx + 2)
    plt.imshow(processed_crop)
    plt.title(f"PARSeq Output: {final_prediction}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import torch
import re
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
from torchvision import transforms as T

# 1. Initialize PARSeq and transformations
DEVICE = "cuda" if torch.torch.cuda.is_available() else "cpu"
parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).eval().to(DEVICE)

parseq_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5)
])

# 2. Helper filters for confidence scoring, decimal styling, and dynamic preprocessing
def filter_by_confidence(raw_str, conf_tensor, threshold=0.40):
    return "".join([char for char, conf in zip(raw_str, conf_tensor) if conf.item() >= threshold])

def post_process_prediction(pred):
    clean = re.sub(r'[^0-9\.]', '', pred)
    if '.' not in clean and len(clean) >= 4:
        clean = clean[:-3] + '.' + clean[-3:]
    return clean

def adaptive_clean_crop(crop_rgb):
    """
    Generalized preprocessing that dynamically handles glare and dust
    by operating strictly within the YOLO bounding box region.
    """
    # [Optional color checking step can be added here in the future]
    
    # 1. Extract Red channel for high-contrast segmentation of red LEDs
    r_channel = crop_rgb[:, :, 0]
    
    # 2. Adaptive thresholding sniper to bypass harsh local daylight/shadow variations
    cleaned = cv2.adaptiveThreshold(
        r_channel, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY_INV, 11, 2
    )
    
    # 3. Apply the custom (2,2) dilation matrix to reconnect faint/broken LED segments
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    final_processed = cv2.dilate(cleaned, kernel, iterations=1)
    
    # CRUCIAL MULTI-CHANNEL CONVERSION: Convert back to 3-channel RGB for PARSeq backend compatibility
    return cv2.cvtColor(final_processed, cv2.COLOR_GRAY2RGB)


# 3. Load the fine-tuned model weights we just generated
new_model = YOLO('/kaggle/working/runs/detect/red_scale_finetune-3/weights/best.pt')

IMAGE_DIR = '/kaggle/working/Red-Scale-Finetune-1/train/images'
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Let's run and display the predictions for the first 5 images
test_files = image_files[:5]
plt.figure(figsize=(15, 5 * len(test_files)))

for idx, file in enumerate(test_files):
    img_path = os.path.join(IMAGE_DIR, file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run inference forcing Class 0 (the scale display) and dropping low-conf noise
    res = new_model(img_rgb, conf=0.60, classes=[0], verbose=False)
    
    boxes = res[0].boxes.xyxy.cpu().numpy()
    if len(boxes) > 0:
        x1, y1, x2, y2 = map(int, boxes[0])
        h_img, w_img = img_rgb.shape[:2]
        
        # Add slight padding for comfortable PARSeq visibility
        pad = int(h_img * 0.03)
        x1_pad = max(0, x1 - pad)
        y1_pad = max(0, y1 - pad)
        x2_pad = min(w_img, x2 + pad)
        y2_pad = min(h_img, y2 + pad)
        
        tight_crop = img_rgb[y1_pad:y2_pad, x1_pad:x2_pad]
        
        # 🔥 THE UPGRADE: Inject the adaptive preprocessing block right here!
        processed_crop = adaptive_clean_crop(tight_crop)
        
        # Run PARSeq sequence matching on the dynamically optimized crop
        with torch.no_grad():
            tensor = parseq_transform(Image.fromarray(processed_crop)).unsqueeze(0).to(DEVICE)
            logits = parseq(tensor)
            decoded_strs, confidences = parseq.tokenizer.decode(logits.softmax(-1))
            
            raw_pred = decoded_strs[0].strip()
            char_confs = confidences[0]

        # Apply confidence parsing and formatting
        filtered_pred = filter_by_confidence(raw_pred, char_confs, threshold=0.40)
        final_prediction = post_process_prediction(filtered_pred)
    else:
        processed_crop = np.zeros((100, 100, 3), dtype=np.uint8)
        final_prediction = "SCALE NOT DETECTED"

    print(f"Image: {file} ---> Predicted Value: {final_prediction}")

    # Plot visual verification
    plt.subplot(len(test_files), 2, 2*idx + 1)
    plt.imshow(res[0].plot())
    plt.title(f"YOLO Box: {file}")
    plt.axis("off")
    
    plt.subplot(len(test_files), 2, 2*idx + 2)
    plt.imshow(processed_crop)
    plt.title(f"PARSeq Output: {final_prediction}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Qnj3uwQmrFh75CdE6blG")
project = rf.workspace("7gamil").project("lp7510-numbers-roi-ocr-gia7i")
version = project.version(4)
dataset = version.download("yolov8")
                

In [ ]:
import os
import cv2
import torch
import re
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
from torchvision import transforms as T

# 1. Pipeline Config & Setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).eval().to(DEVICE)

parseq_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5)
])

def filter_by_confidence(raw_str, conf_tensor, threshold=0.40):
    return "".join([char for char, conf in zip(raw_str, conf_tensor) if conf.item() >= threshold])

def post_process_prediction(pred):
    clean = re.sub(r'[^0-9\.]', '', pred)
    if '.' not in clean and len(clean) >= 4:
        clean = clean[:-3] + '.' + clean[-3:]
    return clean

# Load the new weights we just trained today
new_model = YOLO('/kaggle/working/runs/detect/red_scale_finetune-3/weights/best.pt')

# Path to the true, unseen LP7510 validation folder split
VALID_DIR = "/kaggle/working/LP7510-Numbers-ROI-+-OCR-4/valid/images"

if not os.path.exists(VALID_DIR):
    # Fallback to test or train folder if your export layout was different
    VALID_DIR = "/kaggle/working/LP7510-Validation/train/images"


image_files = [f for f in os.listdir(VALID_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]
test_files = image_files[:5] # Visually verify the first 5 frames

plt.figure(figsize=(15, 5 * len(test_files)))

for idx, file in enumerate(test_files):
    img_path = os.path.join(VALID_DIR, file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Check if the new weights find the green scale housing safely
    res = new_model(img_rgb, conf=0.50, classes=[0], verbose=False)
    boxes = res[0].boxes.xyxy.cpu().numpy()
    
    if len(boxes) > 0:
        x1, y1, x2, y2 = map(int, boxes[0])
        h_img, w_img = img_rgb.shape[:2]
        
        # Apply padding and stable raw crop dilation
        pad = int(h_img * 0.03)
        tight_crop = img_rgb[max(0, y1-pad):min(h_img, y2+pad), max(0, x1-pad):min(w_img, x2+pad)]
        processed_crop = cv2.dilate(tight_crop, cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2)), iterations=1)
        
        with torch.no_grad():
            tensor = parseq_transform(Image.fromarray(processed_crop)).unsqueeze(0).to(DEVICE)
            logits = parseq(tensor)
            decoded_strs, confidences = parseq.tokenizer.decode(logits.softmax(-1))
            
            raw_pred = decoded_strs[0].strip()
            char_confs = confidences[0]
            
        filtered_pred = filter_by_confidence(raw_pred, char_confs, threshold=0.40)
        final_prediction = post_process_prediction(filtered_pred)
    else:
        processed_crop = np.zeros((100, 100, 3), dtype=np.uint8)
        final_prediction = "DETECTION FAILED"
        
    print(f"Image: {file} ---> OCR Predicted: {final_prediction}")
    
    # Plot visual validation
    plt.subplot(len(test_files), 2, 2*idx + 1)
    plt.imshow(res[0].plot())
    plt.title(f"YOLO Box Verification: {file}")
    plt.axis("off")
    
    plt.subplot(len(test_files), 2, 2*idx + 2)
    plt.imshow(processed_crop)
    plt.title(f"PARSeq Text: {final_prediction}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

SUCCESSFUL_RUN_DIR = '/kaggle/working/runs/detect/red_scale_finetune-3/weights/best.pt'

print(f"Loading weights from {SUCCESSFUL_RUN_DIR}...")
model = YOLO(SUCCESSFUL_RUN_DIR)

print("Starting export to TorchScript (mobile optimized)...")
exported_path = model.export(format='torchscript', optimize=True)

print(f"Your mobile-ready file is located at: {exported_path}")